[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/Notebook_02_Gates_and_how_they_work.ipynb)


# Notebook 02 — Quantum Gates and the Circuit Model

Gates are the instructions of a quantum computer. This notebook covers the standard single-qubit Pauli rotations, Hadamard and phase gates, and the CNOT that generates entanglement.

**This is a new notebook, remember to bring your tools.**


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector, array_to_latex
from qiskit_aer import AerSimulator


## The circuit model

A quantum circuit is a time-ordered list of unitary gates, followed by measurement.

- Wires are qubits (and classical bits after measurement).
- Boxes are gates. Controlled gates have a filled dot on the control wire.
- Every closed quantum evolution is a unitary matrix $U$ with $U^\dagger U = I$.

Single-qubit gates are $2\times 2$ unitaries. Two-qubit gates are $4\times 4$ unitaries.


## The X Gate

The **X** gate is the quantum equivalent of the classical **`NOT`** gate. It flips the state of a qubit around the **X axis** of the Bloch sphere by $180°$.

Its action is:

$$X|0\rangle = |1\rangle$$
$$X|1\rangle = |0\rangle$$

As a matrix:

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

It works by swapping the amplitudes $\alpha$ and $\beta$ in the state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$. On the Bloch sphere, $|0\rangle$ sits at the north pole; after the X gate the qubit lands at the south pole $|1\rangle$.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.x(0)
qc.measure(0, 0)
qc.draw("mpl")


In [ ]:
simulator = AerSimulator()
counts = simulator.run(qc, shots=1024).result().get_counts()
plot_histogram(counts)


## The Y Gate

The Y gate rotates the qubit around the **Y axis** of the Bloch sphere by $180°$.

Its action is:

$$Y|0\rangle = i|1\rangle$$
$$Y|1\rangle = -i|0\rangle$$

As a matrix:

$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$

Measurement probabilities match those of $X$ on computational-basis inputs, but the extra phases matter once the qubit is in superposition or is used as a control.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.y(0)
qc.measure(0, 0)
qc.draw("mpl")


In [ ]:
simulator = AerSimulator()
counts = simulator.run(qc, shots=1024).result().get_counts()
plot_histogram(counts)


## The Z Gate

$Z$ is a phase flip around the Bloch $z$-axis:

$$Z|0\rangle = |0\rangle, \qquad Z|1\rangle = -|1\rangle$$

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

On $|0\rangle$ or $|1\rangle$ a $Z$ gate does not change measurement statistics. It *does* change relative phase, which becomes visible after a Hadamard.


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
display(qc.draw("mpl"))
sv = Statevector.from_instruction(qc)
display(array_to_latex(sv, prefix=r"$HZ|0\rangle = $"))
plot_bloch_multivector(sv)


## The Hadamard Gate

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

$$H|0\rangle = |+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}, \qquad
H|1\rangle = |-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}$$

$H$ is its own inverse: $H^2 = I$. That is why "Hadamard, compute, Hadamard" appears in so many algorithms.


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
display(array_to_latex(Operator(qc), prefix="H = "))
plot_bloch_multivector(Statevector.from_instruction(qc))


## Phase gates $S$ and $T$

$$S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix} = \sqrt{Z}, \qquad
T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix} = \sqrt{S}$$

They rotate the Bloch vector around $z$ by $90°$ and $45°$ respectively. Together with $H$ they generate a dense set of single-qubit unitaries.


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
display(qc.draw("mpl"))
plot_bloch_multivector(Statevector.from_instruction(qc))


## CNOT and the two-qubit circuit model

CNOT (CX) flips the target when the control is $|1\rangle$:

$$
\text{CNOT}|c\,t\rangle =
\begin{cases}
|c\,t\rangle & \text{if } c=0\\
|c\rangle|1-t\rangle & \text{if } c=1.
\end{cases}
$$

CNOT is the workhorse for entanglement. Notebook 03 studies that in detail.


In [ ]:
qc = QuantumCircuit(2)
qc.x(0)          # control = qubit 0
qc.cx(0, 1)      # target  = qubit 1
display(qc.draw("mpl"))
sv = Statevector.from_instruction(qc)
print(sv.probabilities_dict())
display(array_to_latex(sv, prefix=r"$|\psi\rangle = $"))


## A tiny reversible circuit

Quantum gates are reversible. Applying the same $X$ twice returns the original bit. The same is true of CNOT.


In [ ]:
qc = QuantumCircuit(2, 2)
qc.x(0)
qc.cx(0, 1)
qc.cx(0, 1)   # undo
qc.x(0)       # undo
qc.measure([0, 1], [0, 1])
display(qc.draw("mpl"))
counts = AerSimulator().run(qc, shots=512).result().get_counts()
plot_histogram(counts)


## Exercises

1. Show numerically that $HZH = X$ by building both circuits and comparing `Operator` matrices.
2. Starting from $|0\rangle$, which sequence lands on $|-\rangle$: $XH$ or $HZ$?
3. Build CNOT with qubit 1 as control and qubit 0 as target. Prepare $|10\rangle$ and check the output.


In [ ]:
# Exercise workspace
